# 3_Transformacion_Silver_Almacenes

## Objetivo
Leer los 20 registros de `bronze.almacenes`, estandarizar los campos de texto y almacenarlos como tabla Delta en `silver.almacenes`.

**Criterio de calidad:** el diagnóstico inicial no identificó nulos ni duplicados en este dataset. Por ello, no se generará una tabla de rechazados artificial. Se conservarán los 20 registros y se validará que los campos obligatorios no contengan valores nulos o vacíos.

La transformación no modifica la tabla Bronze.

## 1. Inicialización

In [ ]:
dbutils.widgets.removeAll()

In [ ]:
from pyspark.sql.functions import col, trim
from pyspark.sql.types import StructType, StructField, IntegerType, StringType

## 2. Parámetros

In [ ]:
dbutils.widgets.text("containerDataLake", "datalake")
dbutils.widgets.text("storageName", "stproyectofinalehs")
dbutils.widgets.text("catalogo", "dbw_proyecto_final_ehs")

containerDataLake = dbutils.widgets.get("containerDataLake")
storageName = dbutils.widgets.get("storageName")
catalogo = dbutils.widgets.get("catalogo")

esquemaBronze = "bronze"
esquemaSilver = "silver"

ruta_silver = f"abfss://{containerDataLake}@{storageName}.dfs.core.windows.net/silver/"

tabla_almacenes_bronze = f"{catalogo}.{esquemaBronze}.almacenes"
tabla_almacenes_silver = f"{catalogo}.{esquemaSilver}.almacenes"

print("Catálogo:", catalogo)
print("Origen:", tabla_almacenes_bronze)
print("Destino:", tabla_almacenes_silver)
print("Ruta Silver:", ruta_silver)

## 3. Lectura desde Bronze

In [ ]:
df_almacenes_bronze = spark.table(tabla_almacenes_bronze)

print("Registros en Bronze:", df_almacenes_bronze.count())
df_almacenes_bronze.printSchema()

## 4. Estandarización de registros

Se aplica `trim()` a los campos de texto para retirar espacios al inicio y al final. Los identificadores y el indicador `es_activo` conservan sus tipos numéricos. Se mantienen las columnas técnicas de trazabilidad.

In [ ]:
df_almacenes_silver = (
    df_almacenes_bronze
    .withColumn("nombre_almacen", trim(col("nombre_almacen")))
    .withColumn("ciudad", trim(col("ciudad")))
    .withColumn("region", trim(col("region")))
)

print("Registros preparados para Silver:", df_almacenes_silver.count())

## 5. Validación de conteos y calidad

In [ ]:
conteo_bronze = df_almacenes_bronze.count()
conteo_silver = df_almacenes_silver.count()

print("Bronze:", conteo_bronze)
print("Silver:", conteo_silver)

nulos_silver = (
    df_almacenes_silver
    .filter(
        col("almacen_id").isNull() |
        col("nombre_almacen").isNull() |
        (col("nombre_almacen") == "") |
        col("ciudad").isNull() |
        (col("ciudad") == "") |
        col("region").isNull() |
        (col("region") == "") |
        col("es_activo").isNull()
    )
    .count()
)

duplicados_silver = (
    df_almacenes_silver
    .groupBy("almacen_id")
    .count()
    .filter(col("count") > 1)
    .count()
)

print("Registros con campos obligatorios nulos/vacíos:", nulos_silver)
print("Identificadores duplicados:", duplicados_silver)

assert conteo_bronze == 20, "Se esperaban 20 registros en Bronze."
assert conteo_silver == conteo_bronze, "Silver debe conservar los 20 registros."
assert nulos_silver == 0, "Se encontraron campos obligatorios nulos o vacíos."
assert duplicados_silver == 0, "Se encontraron almacen_id duplicados."

print("Validaciones correctas.")

## 6. Inspección antes de escribir

In [ ]:
display(df_almacenes_silver.orderBy("almacen_id"))

## 7. Preparar esquema Silver

In [ ]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalogo}.{esquemaSilver}")
print(f"Esquema disponible: {catalogo}.{esquemaSilver}")

## 8. Escritura de tabla Delta

Ejecutar esta celda después de revisar las validaciones y la inspección. `overwrite` reconstruye la tabla Silver si se vuelve a ejecutar el proceso. No modifica Bronze.

In [ ]:
df_almacenes_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .option("path", f"{ruta_silver}almacenes/") \
    .saveAsTable(tabla_almacenes_silver)

print("Tabla creada:", tabla_almacenes_silver)

## 9. Validación final

In [ ]:
df_almacenes_silver_tabla = spark.table(tabla_almacenes_silver)

print("Almacenes Silver:", df_almacenes_silver_tabla.count())
df_almacenes_silver_tabla.printSchema()

assert df_almacenes_silver_tabla.count() == 20, "Se esperaban 20 almacenes en Silver."

print("Validación final correcta.")

## Resultado esperado

- `dbw_proyecto_final_ehs.silver.almacenes`: **20 registros**.
- No se crea tabla de rechazados porque el diagnóstico inicial no encontró registros inválidos.
- Bronze permanece sin modificaciones.
- Los campos de texto se almacenan sin espacios externos.